# BAN6800 Module 4 — SHAP Explainability Notebook

This notebook loads the saved XGBoost model bundle, computes global SHAP values on a reproducible sample, and produces a local explanation. The DataCo dataset is a public proxy, not Shell internal data.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib, shap, matplotlib.pyplot as plt
ROOT=Path.cwd()
# If opened from the notebooks folder, move to the project root.
if ROOT.name=='notebooks': ROOT=ROOT.parent
bundle=joblib.load(ROOT/'models'/'model_bundle.joblib')
df=pd.read_csv(ROOT/'data'/'model_ready.csv.gz', compression='gzip')
df['order_dt']=pd.to_datetime(df['order date (DateOrders)'])
for c,v in {'order_month':df.order_dt.dt.month,'order_dayofweek':df.order_dt.dt.dayofweek,'order_hour':df.order_dt.dt.hour,'order_day':df.order_dt.dt.day,'order_year':df.order_dt.dt.year,'order_quarter':df.order_dt.dt.quarter}.items(): df[c]=v
df=df.sort_values('order_dt').reset_index(drop=True)
test=df.iloc[int(len(df)*.80):].copy()
X=test[bundle['features']]
Xt=bundle['preprocessor'].transform(X)
model=bundle['model']
print('Test rows:',len(test),'| transformed features:',Xt.shape[1])

## Global SHAP importance

The sample is drawn only from the chronological test period. Mean absolute SHAP values summarise the average magnitude of each feature contribution; they do not establish causation.

In [ ]:
rng=np.random.default_rng(42)
idx=np.sort(rng.choice(Xt.shape[0],min(600,Xt.shape[0]),replace=False))
explainer=shap.TreeExplainer(model)
sv=explainer.shap_values(Xt[idx])
importance=pd.DataFrame({'feature':bundle['preprocessor'].get_feature_names_out(),'mean_abs_shap':np.abs(sv).mean(axis=0)}).sort_values('mean_abs_shap',ascending=False)
display(importance.head(20))
shap.summary_plot(sv,Xt[idx],feature_names=bundle['preprocessor'].get_feature_names_out(),plot_type='bar',show=True)

## Local explanation

The next cell selects the highest predicted-risk record in the test period and lists its strongest signed SHAP contributions. A positive SHAP value moves the model output towards the late class; a negative value moves it away. This is a model explanation, not a causal explanation.

In [ ]:
proba=model.predict_proba(Xt)[:,1]
i=int(np.argmax(proba))
local_sv=explainer.shap_values(Xt[i])
local=pd.DataFrame({'feature':bundle['preprocessor'].get_feature_names_out(),'shap_value':np.asarray(local_sv).reshape(-1)})
local['abs_shap']=local.shap_value.abs()
print('Predicted probability:',float(proba[i]),'| threshold:',bundle['threshold'],'| actual label:',int(test.iloc[i]['late_delivery']))
print(test.iloc[i][['Shipping Mode','Days for shipment (scheduled)','Market','Order Region','Customer Segment']])
display(local.sort_values('abs_shap',ascending=False).head(15))

## Interpretation caution

In this dataset, shipping mode and scheduled shipping days dominate the model. The separate feature-ablation results in `reports/feature_ablation.csv` show that removing both sharply reduces discrimination. This suggests dataset-specific structural dependence and must be tested against independent operational data before deployment.